# M03 Feature Probes and Steering


## Probe first, then intervene

We create synthetic feature activations, train a tiny classifier on them, and then add a steering vector to see how the prediction moves.


In [ ]:
import torch
import matplotlib.pyplot as plt

torch.manual_seed(23)

feature_names = ["helpful", "cautious", "concise"]
features = torch.randn(300, 3)
logits_target = 1.2 * features[:, 0] - 0.8 * features[:, 1] + 0.4 * features[:, 2]
labels = (torch.sigmoid(logits_target) > 0.5).float().unsqueeze(1)

probe = torch.nn.Linear(3, 1)
optimizer = torch.optim.Adam(probe.parameters(), lr=0.05)

for step in range(500):
    logits = probe(features)
    loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

weights = probe.weight.detach().flatten()
steering_vector = torch.tensor([0.7, 0.15, -0.1])
example = torch.tensor([[0.1, 0.35, 0.2]])
before = torch.sigmoid(probe(example)).item()
after = torch.sigmoid(probe(example + 0.8 * steering_vector)).item()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(feature_names, weights.tolist(), color=["#1f77b4", "#ff7f0e", "#2ca02c"])
axes[0].set_title("Learned probe weights")
axes[0].axhline(0, color="0.75", linewidth=1)

axes[1].bar(["before", "after"], [before, after], color=["#666666", "#c44e52"])
axes[1].set_ylim(0, 1)
axes[1].set_title("Prediction shift under steering")
plt.tight_layout()

print("Final BCE loss:", float(loss.detach()))
print("Probe weights:", dict(zip(feature_names, [round(v, 3) for v in weights.tolist()])))
print("Example score before steering:", round(before, 3))
print("Example score after steering:", round(after, 3))


## Takeaway

Features become actionable once they support both a readable probe and a controlled intervention. The shift is useful, but it is never free of side effects.
